<a href="https://colab.research.google.com/github/heetaamin/Assignment-3/blob/main/Glass/modelling_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

data_dir = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 3/Glass/'
df = pd.read_csv(data_dir + "glass_clean.csv")
feature_cols = [c for c in df.columns if c != "Type"]
df.shape

Mounted at /content/drive


(213, 10)

# Split the data in train, and validation once

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[feature_cols], df["Type"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=42
)

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

Train: 136  Val: 34  Test: 43


# Doing a per class check

In [ ]:
CLASS_NAMES = {1: "bldg_float", 2: "bldg_nonfloat", 3: "vehicle_float",
               5: "containers", 6: "tableware", 7: "headlamps"}

split_summary = pd.DataFrame({
    "train": y_train.value_counts().sort_index(),
    "val": y_val.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
}).fillna(0).astype(int)
split_summary["total"] = split_summary.sum(axis=1)
split_summary.index = [f"{i} ({CLASS_NAMES[i]})" for i in split_summary.index]
split_summary

,train,val,test,total
1 (bldg_float),44,11,14,69
2 (bldg_nonfloat),49,12,15,76
3 (vehicle_float),11,3,3,17
5 (containers),8,2,3,13
6 (tableware),6,1,2,9
7 (headlamps),18,5,6,29


# Saving this split

In [ ]:
train_df = X_train.copy(); train_df["Type"] = y_train
val_df   = X_val.copy();   val_df["Type"]   = y_val
test_df  = X_test.copy();  test_df["Type"]  = y_test

train_df.to_csv(data_dir + "split_train.csv", index=False)
val_df.to_csv(data_dir + "split_val.csv", index=False)
test_df.to_csv(data_dir + "split_test.csv", index=False)

print("Saved split_train.csv, split_val.csv, split_test.csv to Drive.")
print("Every later notebook (baseline, incremental) loads these three files directly — never re-splits.")

Saved split_train.csv, split_val.csv, split_test.csv to Drive.
Every later notebook (baseline, incremental) loads these three files directly — never re-splits.


# Standardise the training data only

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Encoding the target lables in a minority to majority order

In [ ]:
MINORITY_TO_MAJORITY_LABELS = [6, 5, 3, 7, 1, 2]  # tableware, containers, vehicle_float, headlamps, bldg_float, bldg_nonfloat
label_to_index = {label: i for i, label in enumerate(MINORITY_TO_MAJORITY_LABELS)}
index_to_label = {i: label for label, i in label_to_index.items()}

y_train_idx = y_train.map(label_to_index).values
y_val_idx = y_val.map(label_to_index).values
y_test_idx = y_test.map(label_to_index).values

print(label_to_index)

{6: 0, 5: 1, 3: 2, 7: 3, 1: 4, 2: 5}
